# DFS Pipeline — Data Distribution Analysis

Validates all processed parquet files before projections.py is built.  
Run from the project root after `ingest.py`, `rushing.py`, and `passing.py` have completed.

**Checks covered:**
1. Row counts and season coverage
2. Null / sentinel rates per feature
3. dk_points distribution by position
4. Carry share — team-week sum integrity
5. Target share — team-week sum integrity
6. Key rushing metrics (carry_share, YPC, TD rate)
7. Key passing metrics (target_share, catch rate, YPT, QB rates)
8. Lag feature coverage and plausibility
9. Correlation of all features vs dk_points by position
10. Implied team totals and defensive adjustment distributions

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path

pd.set_option('display.max_columns', 60)
pd.set_option('display.float_format', '{:.3f}'.format)

PROCESSED = Path('../data/processed')

# Load all feature files
players  = pd.read_parquet(PROCESSED / 'players.parquet')
rush     = pd.read_parquet(PROCESSED / 'rushing_features.parquet')
passing  = pd.read_parquet(PROCESSED / 'passing_features.parquet')
schedule = pd.read_parquet(PROCESSED / 'schedule.parquet')

rec = passing[passing['position_id'].isin(['WR','TE','RB'])].copy()
qb  = passing[passing['position_id'] == 'QB'].copy()

print('Loaded all files.')

## 1. Row counts and season coverage

In [ ]:
files = {
    'rushing_features': rush,
    'passing_features (receivers)': rec,
    'passing_features (QB)': qb,
}

for name, df in files.items():
    print(f"{name}:")
    print(f"  rows={len(df):,}  cols={len(df.columns)}  seasons={df['season'].min()}–{df['season'].max()}")
    print(f"  positions: {df['position_id'].value_counts().to_dict()}")
    print()

## 2. Null and sentinel rates

In [ ]:
def null_sentinel_report(df, name, lag_cols):
    print(f"=== {name} ===")
    null_pct = (df.isnull().sum() / len(df) * 100).round(2)
    null_nonzero = null_pct[null_pct > 0]
    if len(null_nonzero):
        print("Null rates:")
        print(null_nonzero.to_string())
    else:
        print("  No nulls.")
    print("Lag sentinel (-1) rates:")
    for col in lag_cols:
        if col in df.columns:
            frac = (df[col] == -1).mean()
            print(f"  {col}: {frac:.3f}")
    print()

null_sentinel_report(
    rush, 'Rushing',
    ['carry_share_lag1','yards_per_carry_lag1','rush_td_rate_lag1','snap_pct_lag1']
)
null_sentinel_report(
    rec, 'Receivers',
    ['target_share_lag1','catch_rate_lag1','yards_per_target_lag1','rec_td_rate_lag1']
)
null_sentinel_report(
    qb, 'QBs',
    ['completion_pct_lag1','yards_per_attempt_lag1','pass_td_rate_lag1','int_rate_lag1']
)

## 3. dk_points distribution by position

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
positions = ['QB', 'RB', 'WR', 'TE']

for ax, pos in zip(axes.flat, positions):
    if pos == 'QB':
        data = qb['dk_points'].dropna()
    else:
        data = rec[rec['position_id'] == pos]['dk_points'].dropna()
    
    ax.hist(data, bins=50, edgecolor='none', alpha=0.8)
    ax.axvline(data.mean(), color='red', linestyle='--', label=f'mean={data.mean():.1f}')
    ax.axvline(data.median(), color='orange', linestyle='--', label=f'median={data.median():.1f}')
    ax.set_title(f'{pos} dk_points  (n={len(data):,})')
    ax.set_xlabel('DK Points')
    ax.set_ylabel('Frequency')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.suptitle('dk_points Distribution by Position (2009–2025)', y=1.01, fontsize=13)
plt.show()

print("Summary statistics:")
for pos in positions:
    if pos == 'QB':
        data = qb['dk_points'].dropna()
    else:
        data = rec[rec['position_id'] == pos]['dk_points'].dropna()
    q90 = data.quantile(0.90)
    q99 = data.quantile(0.99)
    print(f"{pos}: mean={data.mean():.1f}  std={data.std():.1f}  90th={q90:.1f}  99th={q99:.1f}  max={data.max():.1f}")

## 4. Carry share — team-week integrity

In [ ]:
rush_sums = rush[rush['rushing_att'] > 0].groupby(['pfr_code','season','week'])['carry_share'].sum()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(rush_sums, bins=50, edgecolor='none', alpha=0.8)
axes[0].axvline(1.0, color='red', linestyle='--', label='1.0 (ideal)')
axes[0].set_title('Carry Share — team-week sum distribution')
axes[0].set_xlabel('Sum of carry_share per team-week')
axes[0].legend()

axes[1].hist(rush_sums[rush_sums < 2], bins=50, edgecolor='none', alpha=0.8)
axes[1].axvline(1.0, color='red', linestyle='--', label='1.0 (ideal)')
axes[1].set_title('Carry Share — zoomed (sum < 2)')
axes[1].set_xlabel('Sum of carry_share per team-week')
axes[1].legend()

plt.tight_layout()
plt.show()

pct_ok = ((rush_sums >= 0.90) & (rush_sums <= 1.10)).mean()
print(f"Team-weeks with sum in [0.90, 1.10]: {pct_ok:.1%}")
print(f"Mean sum: {rush_sums.mean():.3f}  |  Max sum: {rush_sums.max():.3f}")
print("\nOutlier team-weeks (sum > 1.10):")
outliers = rush_sums[rush_sums > 1.10].reset_index()
outliers.columns = ['pfr_code','season','week','carry_share_sum']
print(outliers.sort_values('carry_share_sum', ascending=False).head(10).to_string(index=False))

## 5. Target share — team-week integrity

In [ ]:
tar_sums = rec[rec['receiving_tar'] > 0].groupby(['pfr_code','season','week'])['target_share'].sum()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(tar_sums, bins=50, edgecolor='none', alpha=0.8)
axes[0].axvline(1.0, color='red', linestyle='--', label='1.0 (ideal)')
axes[0].set_title('Target Share — team-week sum distribution')
axes[0].set_xlabel('Sum of target_share per team-week')
axes[0].legend()

axes[1].hist(tar_sums[tar_sums < 1.5], bins=50, edgecolor='none', alpha=0.8)
axes[1].axvline(1.0, color='red', linestyle='--', label='1.0 (ideal)')
axes[1].set_title('Target Share — zoomed (sum < 1.5)')
axes[1].set_xlabel('Sum of target_share per team-week')
axes[1].legend()

plt.tight_layout()
plt.show()

pct_ok = ((tar_sums >= 0.90) & (tar_sums <= 1.10)).mean()
print(f"Team-weeks with sum in [0.90, 1.10]: {pct_ok:.1%}")
print(f"Mean sum: {tar_sums.mean():.3f}  |  Max sum: {tar_sums.max():.3f}")
print("\nNote: sums < 1.0 occur when QBs are excluded (QB rushing/scramble non-target plays)")
print("Outlier team-weeks (sum > 1.10):")
outliers = tar_sums[tar_sums > 1.10].reset_index()
outliers.columns = ['pfr_code','season','week','target_share_sum']
print(outliers.sort_values('target_share_sum', ascending=False).head(10).to_string(index=False))

## 6. Key rushing metrics — distributions

In [ ]:
# Focus on players who actually rushed (att >= 1)
active_rush = rush[rush['rushing_att'] >= 1].copy()

metrics = [
    ('carry_share',     'Carry Share (player / team carries)', (0, 1)),
    ('yards_per_carry', 'Yards Per Carry', (-5, 15)),
    ('rush_td_rate',    'Rush TD Rate (per carry)', (0, 0.5)),
    ('snap_pct',        'Snap % (rushers with att >= 1)', (0, 1)),
]

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

for ax, (col, title, xlim) in zip(axes.flat, metrics):
    data = active_rush[col].dropna()
    data_clipped = data.clip(*xlim)
    ax.hist(data_clipped, bins=60, edgecolor='none', alpha=0.8)
    ax.axvline(data.mean(), color='red', linestyle='--', label=f'mean={data.mean():.3f}')
    ax.axvline(data.median(), color='orange', linestyle='--', label=f'median={data.median():.3f}')
    ax.set_title(title)
    ax.set_xlabel(col)
    ax.set_ylabel('Frequency')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.suptitle('Rushing Metrics — Players with ≥1 Carry (2009–2025)', y=1.01, fontsize=13)
plt.show()

print(active_rush[['carry_share','yards_per_carry','rush_td_rate','snap_pct']].describe().round(3))

## 7. Key passing metrics — distributions

In [ ]:
active_rec = rec[rec['receiving_tar'] >= 1].copy()
active_qb  = qb[qb['passing_att'] >= 10].copy()   # 10+ att to exclude gadget plays

fig, axes = plt.subplots(2, 4, figsize=(20, 8))

rec_metrics = [
    ('target_share',     'Target Share', (0, 0.5)),
    ('catch_rate',       'Catch Rate',   (0, 1)),
    ('yards_per_target', 'Yards/Target', (0, 20)),
    ('rec_td_rate',      'Rec TD Rate',  (0, 0.3)),
]
qb_metrics = [
    ('completion_pct',    'Completion %',    (0.3, 1)),
    ('yards_per_attempt', 'Yards/Attempt',   (3, 12)),
    ('pass_td_rate',      'Pass TD Rate',    (0, 0.15)),
    ('int_rate',          'INT Rate',        (0, 0.10)),
]

for ax, (col, title, xlim) in zip(axes[0], rec_metrics):
    data = active_rec[col].dropna().clip(*xlim)
    ax.hist(data, bins=50, edgecolor='none', alpha=0.8, color='steelblue')
    ax.axvline(active_rec[col].mean(), color='red', linestyle='--',
               label=f'mean={active_rec[col].mean():.3f}')
    ax.set_title(f'Receivers — {title}')
    ax.legend(fontsize=7)

for ax, (col, title, xlim) in zip(axes[1], qb_metrics):
    data = active_qb[col].dropna().clip(*xlim)
    ax.hist(data, bins=50, edgecolor='none', alpha=0.8, color='darkorange')
    ax.axvline(active_qb[col].mean(), color='red', linestyle='--',
               label=f'mean={active_qb[col].mean():.3f}')
    ax.set_title(f'QB — {title}')
    ax.legend(fontsize=7)

plt.tight_layout()
plt.suptitle('Passing / Receiving Metrics (2009–2025)', y=1.01, fontsize=13)
plt.show()

print("=== Receiver stats (tar >= 1) ===")
print(active_rec[['target_share','catch_rate','yards_per_target','rec_td_rate']].describe().round(3))
print("\n=== QB stats (att >= 10) ===")
print(active_qb[['completion_pct','yards_per_attempt','pass_td_rate','int_rate']].describe().round(3))

## 8. Lag feature plausibility — lag1 vs current value

In [ ]:
# Scatter lag1 vs current for primary usage features (non-sentinel rows only)
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Carry share
r = rush[(rush['carry_share_lag1'] != -1) & (rush['rushing_att'] > 0)].sample(5000, random_state=42)
axes[0].scatter(r['carry_share_lag1'], r['carry_share'], alpha=0.15, s=8)
axes[0].set_xlabel('carry_share_lag1'); axes[0].set_ylabel('carry_share (current)')
axes[0].set_title(f'Carry Share: lag1 vs current  r={r["carry_share_lag1"].corr(r["carry_share"]):.3f}')

# Target share
r2 = rec[(rec['target_share_lag1'] != -1) & (rec['receiving_tar'] > 0)].sample(5000, random_state=42)
axes[1].scatter(r2['target_share_lag1'], r2['target_share'], alpha=0.15, s=8, color='steelblue')
axes[1].set_xlabel('target_share_lag1'); axes[1].set_ylabel('target_share (current)')
axes[1].set_title(f'Target Share: lag1 vs current  r={r2["target_share_lag1"].corr(r2["target_share"]):.3f}')

# YPA for QB
r3 = qb[(qb['yards_per_attempt_lag1'] != -1) & (qb['passing_att'] > 10)].sample(min(3000, len(qb)), random_state=42)
axes[2].scatter(r3['yards_per_attempt_lag1'], r3['yards_per_attempt'], alpha=0.2, s=8, color='darkorange')
axes[2].set_xlabel('yards_per_attempt_lag1'); axes[2].set_ylabel('yards_per_attempt (current)')
axes[2].set_title(f'QB YPA: lag1 vs current  r={r3["yards_per_attempt_lag1"].corr(r3["yards_per_attempt"]):.3f}')

plt.tight_layout()
plt.suptitle('Lag1 vs Current — Usage Stability', y=1.02, fontsize=13)
plt.show()

## 9. Feature correlation with dk_points by position

In [ ]:
def corr_with_target(df, feature_cols, target='dk_points', title=''):
    corrs = {}
    for col in feature_cols:
        if col not in df.columns:
            continue
        sub = df[(df[col] != -1) & df[col].notna() & df[target].notna()]
        if len(sub) < 100:
            continue
        corrs[col] = sub[col].corr(sub[target])
    s = pd.Series(corrs).sort_values(key=abs, ascending=False)
    
    fig, ax = plt.subplots(figsize=(10, max(4, len(s) * 0.35)))
    colors = ['steelblue' if v >= 0 else 'tomato' for v in s.values]
    ax.barh(s.index[::-1], s.values[::-1], color=colors[::-1])
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_xlabel('Pearson r with dk_points')
    ax.set_title(title)
    plt.tight_layout()
    plt.show()
    return s

# RB
rb = rush[rush['position_id'] == 'RB']
rb_features = ['carry_share','yards_per_carry','rush_td_rate','snap_pct',
               'carry_share_lag1','yards_per_carry_lag1','rush_td_rate_lag1','snap_pct_lag1',
               'rushing_att','rushing_att_lag1']
print("=== RB feature correlations with dk_points ===")
rb_corrs = corr_with_target(rb, rb_features, title='RB — Feature Correlation with dk_points')
print(rb_corrs.round(3).to_string())

In [ ]:
# WR
wr = rec[rec['position_id'] == 'WR']
rec_features = ['target_share','catch_rate','yards_per_target','rec_td_rate','snap_pct',
                'target_share_lag1','catch_rate_lag1','yards_per_target_lag1','rec_td_rate_lag1',
                'snap_pct_lag1','receiving_tar','receiving_tar_lag1']
print("=== WR feature correlations with dk_points ===")
wr_corrs = corr_with_target(wr, rec_features, title='WR — Feature Correlation with dk_points')
print(wr_corrs.round(3).to_string())

In [ ]:
# TE
te = rec[rec['position_id'] == 'TE']
print("=== TE feature correlations with dk_points ===")
te_corrs = corr_with_target(te, rec_features, title='TE — Feature Correlation with dk_points')
print(te_corrs.round(3).to_string())

In [ ]:
# QB
qb_features = ['completion_pct','yards_per_attempt','pass_td_rate','int_rate','snap_pct',
               'completion_pct_lag1','yards_per_attempt_lag1','pass_td_rate_lag1','int_rate_lag1',
               'passing_att','passing_att_lag1']
print("=== QB feature correlations with dk_points ===")
qb_corrs = corr_with_target(qb, qb_features, title='QB — Feature Correlation with dk_points')
print(qb_corrs.round(3).to_string())

## 10. Team context — implied totals and defensive adjustments

In [ ]:
ctx_path = Path('../data/processed/team_game_context.parquet')
if ctx_path.exists():
    ctx = pd.read_parquet(ctx_path)
    
    fig, axes = plt.subplots(2, 3, figsize=(18, 8))
    cols = [
        ('implied_team_total',  'Implied Team Total',           (10, 45)),
        ('expected_pass_att',   'Expected Pass Attempts',       (15, 55)),
        ('expected_rush_att',   'Expected Rush Attempts',       (10, 55)),
        ('pass_def_adj',        'Pass Defense Adjustment',      (0.7, 1.3)),
        ('rush_def_adj',        'Rush Defense Adjustment',      (0.7, 1.3)),
        ('game_script_factor',  'Game Script Factor',           (-0.2, 0.2)),
    ]
    for ax, (col, title, xlim) in zip(axes.flat, cols):
        data = ctx[col].dropna().clip(*xlim)
        ax.hist(data, bins=20, edgecolor='none', alpha=0.8)
        ax.axvline(ctx[col].mean(), color='red', linestyle='--',
                   label=f'mean={ctx[col].mean():.2f}')
        ax.set_title(title)
        ax.legend(fontsize=8)
    
    plt.tight_layout()
    plt.suptitle(f'Team Game Context — Week {ctx["week"].iloc[0]} Season {ctx["season"].iloc[0]}', y=1.01)
    plt.show()
    
    print(ctx[['pfr_code','opponent_pfr_code','implied_team_total','expected_pass_att',
               'expected_rush_att','pass_def_adj','rush_def_adj']].to_string(index=False))
else:
    print("team_game_context.parquet not found — run team_scoring.py first.")

## Summary — Data Readiness Checklist

Run this cell last to get a pass/fail summary before building projections.py.

In [ ]:
checks = []

# Season filter applied
checks.append(('Season filter >= 2009',
               rush['season'].min() >= 2009 and passing['season'].min() >= 2009))

# Row counts reasonable
checks.append(('Rushing rows > 50K', len(rush) > 50_000))
checks.append(('Passing rows > 80K', len(passing) > 80_000))

# No critical nulls in target column
checks.append(('dk_points null rate == 0', rush['dk_points'].isnull().sum() == 0))
checks.append(('Passing dk_points null rate == 0', passing['dk_points'].isnull().sum() == 0))

# Carry share integrity
carry_sums = rush[rush['rushing_att'] > 0].groupby(['pfr_code','season','week'])['carry_share'].sum()
checks.append(('Carry share team-week sums 90%+ within [0.9, 1.1]',
               ((carry_sums >= 0.90) & (carry_sums <= 1.10)).mean() > 0.90))

# Target share integrity
rec_local = passing[passing['position_id'].isin(['WR','TE','RB'])]
tar_sums = rec_local[rec_local['receiving_tar'] > 0].groupby(['pfr_code','season','week'])['target_share'].sum()
checks.append(('Target share team-week sums 90%+ within [0.9, 1.1]',
               ((tar_sums >= 0.90) & (tar_sums <= 1.10)).mean() > 0.90))

# Plausible metric ranges
checks.append(('target_share max < 1.0', passing['target_share'].max() < 1.0))
checks.append(('QB completion_pct mean in [0.55, 0.70]',
               0.55 < qb[qb['passing_att'] >= 10]['completion_pct'].mean() < 0.70))
checks.append(('QB YPA mean in [6.0, 8.5]',
               6.0 < qb[qb['passing_att'] >= 10]['yards_per_attempt'].mean() < 8.5))

print("=== DATA READINESS CHECKLIST ===")
all_pass = True
for label, result in checks:
    status = 'PASS' if result else 'FAIL'
    if not result:
        all_pass = False
    print(f"  [{status}] {label}")

print()
if all_pass:
    print("All checks passed. Ready to build projections.py.")
else:
    print("Some checks FAILED. Review above before proceeding to projections.py.")